# DATA304 Final Project — Five Reviews to One Product

**End-to-end, single-notebook implementation**: metadata-only GPT-4o mini aspect extraction → ColBERT-style token-level late-interaction retriever (ranking, multi-aspect supervision, and gold review-pair classification) → cross-encoder reranker → validation-calibrated pseudo-labeling → updated retrieval index → top-10 Kaggle predictions. BM25 and pretrained dense baselines are included for the required comparisons.

**Dataset**: `catalog.json`, `train_gold.json`, `test_queries.json`. An accompanying `category_taxonomy.txt` is used to check gold category paths. Use `dummy_baseline.ipynb` to check the final CSV layout. ColBERT scores each review separately and averages the five MaxSim scores. The reranker joins the reviews before applying its input length limit.

**Important compliance notes**

- **Never** send review text, summaries, or rewrites to external LLM APIs. The GPT step reads **catalog metadata only**. This notebook does not use web data or Amazon-finetuned checkpoints.
- The 1,000-call API ceiling is enforced across restarts using a durable append-only *attempt log*. Separate raw input and output JSONL files are saved for course verification. An API attempt counts even if it fails.
- The selected backbone, `distilbert/distilbert-base-uncased`, is a general pretrained model described as trained on BookCorpus and Wikipedia. Check the exact model card and course policy before final submission: [model card](https://huggingface.co/distilbert/distilbert-base-uncased).
- Place the three official JSON files together in the project release folder or beside the notebook. Set `DATA_DIR` for a different location. Place `category_taxonomy.txt` in the same folder to enable the category-path check.
- An NVIDIA GPU is strongly recommended. Full catalog encoding, token ANN, and 20k-query pseudo-labeling are computationally expensive. Use the debug controls first; disable them for the final Kaggle run.
- The course evaluates **Recall@10**, so top-10 coverage matters more than precise top-1 ranking. Validate the exact CSV structure against the supplied `dummy_baseline.ipynb`.

Notebook order: **0. Setup → 1. Data → 2. GPT aspects → 3. Sampling/model → 4. ColBERT training → 5. Cross-encoder → 6. ANN+validation → 7. Pseudo-labeling → 8. Baselines → 9. Prediction.** Optional hierarchical classification/GNN is intentionally left as an extension, not silently represented as implemented.

In [ ]:
## 0. Environment and configuration

Run the install cell once if required, then restart the kernel if it changes PyTorch/Transformers. Configuration, output directories, random seeds, and all paths are collected here. The pipeline is **not executed automatically when the file is opened**; executing cells incurs GPU time, and the GPT stage may spend API credits.

In [ ]:
!nvidia-smi # OK if NVIDIA L4 is shown
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))
%pip install -q transformers datasets # only what you need

In [ ]:
# 검색기와 재랭커에는 transformers, 상품 토큰 검색에는 FAISS, 단어 기반 비교 실험에는 BM25를 쓴다.
# 현재 환경에서 빠진 패키지만 설치하므로 노트북을 다시 열 때도 이 셀을 그대로 실행할 수 있다.
import sys,subprocess,importlib.util
needed={'transformers':'transformers>=4.46,<5', 'openai':'openai>=1.40,<4',
        'faiss':'faiss-cpu>=1.8', 'rank_bm25':'rank-bm25>=0.2.2',
        'tqdm':'tqdm>=4.66'}
missing=[pip_name for module,pip_name in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing packages:',missing)
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
assert importlib.util.find_spec('torch') is not None, 'Install a CUDA-compatible PyTorch build for your GPU first.'

In [ ]:
# 이 셀에서 실행 장치, 재현용 시드, 데이터 경로와 이후 단계가 공유할 설정을 준비한다.
import os, json, math, random, re, time, itertools
from collections import Counter, defaultdict
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

# 상품 분할, 오답 추출, 모델 초기화의 시드를 고정해 pair loss 실험의 출발 조건을 맞춘다.
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# DATA_DIR가 None이면 데이터 폴더를 자동 탐색한다. 결과와 모델 가중치는 WORK_DIR 아래에 저장한다.
DATA_DIR = None
WORK_DIR = Path('data304_outputs')
WORK_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = WORK_DIR / 'checkpoints'; CKPT_DIR.mkdir(exist_ok=True)

# query_tokens는 ColBERT가 리뷰 한 개에서 읽을 길이, metadata_tokens는 상품 하나에서 읽을 길이다.
# cross_tokens는 재랭커가 리뷰 다섯 개와 상품을 함께 읽을 때 적용하는 전체 길이 한도다.
CFG = dict(
    backbone='distilbert/distilbert-base-uncased', dimension=64,
    query_tokens=72, metadata_tokens=112, cross_tokens=448,
    cross_tokens_per_review=58,
    val_fraction=0.10, train_limit=None, val_eval_limit=350,
    negative_count=8, negative_mode='mixed',  # mixed는 카테고리 오답 포함, random은 무작위만 사용
    rank_temperature=1.0, meta_aspect_weight=0.15, review_aspect_weight=0.05,
    # pair_temperature는 리뷰 간 내적을 logit으로 바꿀 때 쓰고, pair_loss_weight는 손실의 비중을 정한다.
    # 같은 시드에서 pair_loss_enabled만 바꿔 학습하면 리뷰 쌍 분류의 효과를 비교할 수 있다.
    pair_loss_enabled=True, pair_loss_weight=0.1, pair_temperature=0.1,
    aspect_enabled=True, aspect_epochs=1, aspect_batch=24,
    retriever_epochs=2, retriever_batch=2, cross_epochs=2, cross_batch=2,
    pseudo_epochs=1, pseudo_loss_weight=0.20,
    lr_retriever=2e-5, lr_cross=2e-5, lr_aspect=2e-5,
    max_grad_norm=1.0,
    # 한 요청에 상품 64개를 담으면 5만 개 상품에 최소 782회가 필요하다.
    use_gpt_aspects=True, gpt_model='gpt-4o-mini', gpt_batch_size=64,
    max_llm_attempts=1000, gpt_max_output_tokens=10000,
    # ANN으로 토큰 이웃을 찾고 ann_pool개의 후보를 모은 뒤, MaxSim 상위 candidate_topk개를 재랭커에 넘긴다.
    ann_nlist=1024, ann_nprobe=32, ann_per_token=48,
    ann_pool=450, candidate_topk=50, cross_infer_batch=32,
    index_doc_batch=96, index_add_docs=500,
    # 검증에서 확보할 최소 표본 수와 목표 정밀도를 정해 pseudo label 채택 기준을 고른다.
    target_pseudo_precision=0.90, calibration_min_examples=15,
    min_review_agreement=2, max_colbert_rank=5,
    pseudo_limit=None, pseudo_max_selected=5000,
    run_pseudo=True, run_baselines=True, baseline_eval_limit=200,
    student_id='YOUR_STUDENT_ID',
)

# with 블록 안에서 사용할 연산 정밀도를 정한다. CUDA가 bfloat16을 지원할 때 혼합 정밀도를 켠다.
def amp_context():
    return torch.autocast(device_type='cuda', dtype=torch.bfloat16) if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported() else nullcontext()

# 분할 정보나 평가 결과 같은 파이썬 객체를 들여쓰기한 UTF-8 JSON 파일로 저장한다.
def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')

# UTF-8 JSON 파일을 읽어 원래 자료 구조로 돌려준다. 파일 앞에 BOM이 있어도 읽을 수 있다.
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))

## 1. Data ingestion and leakage-safe split

Each catalog entry supplies an `item_id` and four text fields. Each query supplies a `query_id` and five review objects. The model reads the `text` field of each review in order. Gold queries also supply the target `item_id` and a `category_path`.

The split groups gold queries by product ID. Training uses the category paths from the training products. Held-out gold queries provide retrieval metrics and calibrate the threshold for assigning pseudo labels to test queries.

In [ ]:
# 카탈로그 한 행은 검색할 상품 하나다. 쿼리 한 행은 같은 상품에 달린 리뷰 다섯 개다.
# gold에는 찾아야 할 상품 ID가 주어지고, 테스트에서는 이 리뷰들로 상품 ID의 순위를 예측한다.
DATA_FILES = ('catalog.json', 'train_gold.json', 'test_queries.json')

# 세 JSON이 함께 있는 폴더를 찾는다. DATA_DIR을 지정하면 해당 폴더를 사용한다.
def locate_data_dir():
    root = Path.cwd()
    if DATA_DIR is not None:
        candidates = [Path(DATA_DIR)]
    else:
        release = root / 'project_release'
        candidates = [root, release, release / 'project_release', root.parent, root.parent / 'project_release']
    for folder in candidates:
        if all((folder / name).is_file() for name in DATA_FILES):
            return folder
    raise FileNotFoundError(f"Expected {', '.join(DATA_FILES)} in one folder. Checked {candidates}")


# 상품 인코더와 재랭커가 읽을 문자열을 만든다. 네 필드는 모두 문자열이고 일부 값은 비어 있다.
# 값이 있는 필드만 이름을 붙여 이어 주면 설명이 빈 상품도 상품명 등 남은 정보로 검색할 수 있다.
def metadata_text(item):
    fields = (('title', 'Product'), ('store', 'Store'), ('features', 'Features'), ('description', 'Description'))
    parts = []
    for field, label in fields:
        value = item[field].strip()
        if value:
            parts.append(f'{label}: {value}')
    return ' [SEP] '.join(parts) or 'Product metadata not available'


# gold의 '상위 > ... > 말단' 경로를 단계별 튜플로 만든다. path[:-1]은 마지막 단계를 뺀 부모 경로다.
def normalize_path(path):
    return tuple(level.strip() for level in path.split('>') if level.strip())


# query_id는 리뷰 묶음의 식별자다. gold의 item_id가 이 묶음과 카탈로그 상품을 연결하는 정답이다.
def normalize_query(row, *, is_gold):
    reviews = row['reviews']
    if len(reviews) != 5:
        raise ValueError(f"Expected five reviews for {row['query_id']}, got {len(reviews)}.")
    # 리뷰 하나에는 text와 부가 정보(rating, helpful_vote, verified_purchase, timestamp)가 들어 있다.
    # 검색 모델에는 본문만 쓰므로 text 다섯 개를 원래 순서로 꺼낸다.
    query = {
        'query_id': row['query_id'],
        'reviews': [review['text'].strip() for review in reviews],
    }
    if is_gold:
        query['item_id'] = row['item_id']
        query['category_path'] = normalize_path(row['category_path'])
    return query


DATA_PATH = locate_data_dir()
# 세 JSON의 최상위는 모두 리스트다. 카탈로그의 원래 순서로 ID와 텍스트를 두면
# 나중에 검색 인덱스가 반환한 행 번호 i를 ITEM_IDS[i]로 바꿔 제출할 수 있다.
CATALOG = {item['item_id']: item for item in read_json(DATA_PATH / 'catalog.json')}
ITEM_IDS = list(CATALOG)
ITEM_TO_IDX = {item_id: index for index, item_id in enumerate(ITEM_IDS)}
CAT_TEXTS = [metadata_text(CATALOG[item_id]) for item_id in ITEM_IDS]

# 원본 리뷰 객체는 파일을 읽는 동안만 쓰고, 이후 셀은 정리된 본문 리스트를 공유한다.
# TEST를 파일 순서대로 만들기 때문에 마지막 CSV도 같은 순서로 기록된다.
GOLD_ALL = [normalize_query(row, is_gold=True) for row in read_json(DATA_PATH / 'train_gold.json')]
TEST = [normalize_query(row, is_gold=False) for row in read_json(DATA_PATH / 'test_queries.json')]
assert len(CATALOG) == 50_000, f'Catalog size differs from the assignment: {len(CATALOG)}'
assert len({row['query_id'] for row in GOLD_ALL}) == len(GOLD_ALL)
assert len({row['query_id'] for row in TEST}) == len(TEST)
assert all(row['item_id'] in CATALOG for row in GOLD_ALL)

# 검증 상품을 먼저 뽑고 해당 상품의 리뷰 묶음을 모두 VAL에 둔다.
# 이 단위로 나누면 같은 상품의 다른 리뷰가 TRAIN을 통해 모델에 노출되는 일을 막을 수 있다.
unique_gold_items = sorted({row['item_id'] for row in GOLD_ALL})
rng = np.random.default_rng(SEED)
rng.shuffle(unique_gold_items)
n_val = max(1, int(round(len(unique_gold_items) * CFG['val_fraction'])))
val_items = set(unique_gold_items[:n_val])
TRAIN = [row for row in GOLD_ALL if row['item_id'] not in val_items]
VAL = [row for row in GOLD_ALL if row['item_id'] in val_items]
assert val_items.isdisjoint(row['item_id'] for row in TRAIN)
if CFG['train_limit'] is not None:
    TRAIN = TRAIN[:CFG['train_limit']]

# 이 대응표로 학습 상품의 category 레이블을 보정하고, 정답과 종류가 비슷한 랭킹 오답을 뽑는다.
TRAIN_CATEGORY = {row['item_id']: row['category_path'] for row in TRAIN if row['category_path']}
save_json({
    'seed': SEED,
    'train_query_ids': [row['query_id'] for row in TRAIN],
    'val_query_ids': [row['query_id'] for row in VAL],
}, WORK_DIR / 'split.json')
print('catalog:', len(CATALOG), 'gold train:', len(TRAIN), 'gold val:', len(VAL), 'unlabeled test:', len(TEST))

# 별도 분류표가 있으면 학습 상품의 경로와 대조한다. 학습에 쓰는 경로의 출처는 위의 train_gold.json이다.
taxonomy_file = DATA_PATH / 'category_taxonomy.txt'
TAXONOMY = set()
if taxonomy_file.is_file():
    with taxonomy_file.open(encoding='utf-8-sig') as lines:
        for line in lines:
            # TXT에 |나 /가 쓰인 경우에도 gold와 같은 단계별 튜플로 바꾼다.
            path = normalize_path(re.sub(r'[|/]', '>', line))
            if path:
                TAXONOMY.add(path)
    matched = sum(path in TAXONOMY for path in TRAIN_CATEGORY.values())
    print('taxonomy paths:', len(TAXONOMY), '| matching train products:', matched, '/', len(TRAIN_CATEGORY))
    if TAXONOMY and TRAIN_CATEGORY and not matched:
        print('WARNING: inspect the category path format before using category-aware negatives.')
else:
    print('Category taxonomy check skipped (category_taxonomy.txt was not supplied).')

## 2. Metadata-only GPT-4o mini aspect labeling

Generate structured **Category, Brand, Color, Material, Size, Features** labels using metadata only. Unknown attributes remain `unknown`. This stage uses **one API request per batch**, persists exact request payloads and raw responses as separate JSONL files, and supports resume without repeating completed products. The SDK's automatic retries are disabled so each model request has its own attempt record. Completed batches are checkpointed individually and merged into the original JSON cache after labeling finishes. Gold train categories override generated categories; validation categories do not leak into training.

The generated labels are *weak supervision*: verify a small manual sample and do not treat inferred material, brand, or size as ground truth. A malformed or incomplete batch is retried only if API budget permits. When an API key is unavailable, you may set `use_gpt_aspects=False` **only if you already have a complete aspects cache**.

In [ ]:
# 카탈로그에는 상품별 속성 정답표가 없으므로 메타데이터에서 여섯 속성을 추출해 보조 학습의 레이블로 쓴다.
# 속성은 상품 ID별로 캐시하고, 요청·응답은 별도 파일에 기록해 어느 상품까지 처리했는지 추적한다.
ASPECT_KEYS = ['category','brand','color','material','size','features']
ASPECT_CACHE = WORK_DIR/'metadata_aspects.json'
ASPECT_BATCH_DIR = WORK_DIR/'metadata_aspect_checkpoints'
LLM_ATTEMPTS = WORK_DIR/'llm_attempts.jsonl'
LLM_INPUTS = WORK_DIR/'llm_inputs.jsonl'
LLM_OUTPUTS = WORK_DIR/'llm_outputs.jsonl'

# 호출 시도·입력·응답 기록 하나를 JSONL 끝에 붙이고 디스크에 반영해 재시작 시에도 횟수를 추적한다.
def append_jsonl(path, obj):
    with Path(path).open('a',encoding='utf-8') as f:
        f.write(json.dumps(obj,ensure_ascii=False)+'\n')
        f.flush(); os.fsync(f.fileno())

# GPT에 보낼 상품 하나를 구성한다. 카탈로그의 네 문자열을 필드별 글자 수 한도까지 잘라 배치 요청에 담는다.
# item_id는 응답을 같은 상품에 연결할 때 쓴다. 모델이 검색에 쓸 리뷰 본문은 별도 쿼리 파일에 있다.
def metadata_for_llm(item_id):
    item = CATALOG[item_id]
    return {
        'item_id': item_id,
        'title': item['title'].strip()[:200],
        'store': item['store'].strip()[:100],
        'features': item['features'].strip()[:700],
        'description': item['description'].strip()[:900],
    }

# GPT가 메타데이터에서 추출할 속성명, JSON 응답 형태, 알 수 없는 값의 표기 방법을 지정한다.
SYSTEM_PROMPT = (
    'Extract attributes only from the provided PRODUCT METADATA. No reviews are present. '
    'Return a JSON object with exactly the key "items", a list of objects with '
    '"item_id", "category", "brand", "color", "material", "size", "features". '
    'The first five attributes must be strings; features must be a list of up to five short strings. '
    'If an attribute cannot be reasonably inferred from the metadata, write "unknown" '
    '(or [] for features). Prefer concise canonical category names. Do not invent unsupported '
    'technical claims, specifications, or model numbers; no commentary or markdown.'
)

# 카탈로그의 features는 설명 문자열이다. GPT는 상품 메타데이터에서 특성 이름을 뽑아 features 목록으로 돌려준다.
# 이 함수는 응답 표현을 정리해 단일값의 결측은 unknown, 특성 목록의 결측은 빈 리스트로 맞춘다.
def parse_aspects(x):
    d = {}
    for key in ASPECT_KEYS:
        val = x.get(key, [] if key=='features' else 'unknown')
        if key=='features':
            if isinstance(val,str): val=[] if val.lower()=='unknown' else [val]
            if not isinstance(val,list): val=[]
            d[key] = [str(y).strip().lower() for y in val[:5] if str(y).strip().lower() not in ('','unknown','none')]
        else:
            d[key] = str(val).strip().lower() if val is not None else 'unknown'
            if not d[key] or d[key] in ('none','n/a','null'): d[key]='unknown'
    return d

# 속성 레이블을 배치 체크포인트나 최종 캐시 파일에 저장한다.
def save_aspect_cache(path, labels):
    # 임시 파일을 끝까지 쓴 후 교체하므로 저장 도중 중단되어도 직전 캐시를 읽을 수 있다.
    temporary = path.with_suffix('.tmp')
    save_json(labels, temporary)
    temporary.replace(path)

# 최종 캐시와 완료된 배치 체크포인트를 합쳐 남은 상품 ID와 지금까지의 호출 횟수를 복원한다.
aspect_cache = read_json(ASPECT_CACHE) if ASPECT_CACHE.exists() else {}
completed_batches = sorted(ASPECT_BATCH_DIR.glob('batch_[0-9]*.json'))
for checkpoint in completed_batches:
    aspect_cache.update(read_json(checkpoint))
pending = [item_id for item_id in ITEM_IDS if item_id not in aspect_cache]
attempts = sum(1 for _ in LLM_ATTEMPTS.open(encoding='utf-8')) if LLM_ATTEMPTS.exists() else 0

# 미처리 상품이 있으면 API 사용 설정·키·남은 호출 예산을 확인하고 배치 단위로 요청한다.
if pending:
    if not CFG['use_gpt_aspects']:
        raise RuntimeError(f'{len(pending)} metadata items lack cached aspects. Enable the GPT stage or supply a complete cache.')
    if not os.getenv('OPENAI_API_KEY'):
        raise RuntimeError('Set OPENAI_API_KEY in the environment before starting metadata-only GPT calls. No call was made.')
    from openai import OpenAI
    # SDK 자동 재시도를 꺼서 노트북이 기록하는 호출 시도와 별개로 요청이 반복되지 않게 한다.
    api = OpenAI(max_retries=0)
    batches = [pending[i:i+CFG['gpt_batch_size']] for i in range(0,len(pending),CFG['gpt_batch_size'])]
    request_limit = min(1000, CFG['max_llm_attempts'])
    if attempts + len(batches) > request_limit:
        raise RuntimeError(f'Insufficient API-call budget: used {attempts}, need >= {len(batches)} more. Reduce failed calls or obtain course guidance; do not exceed 1,000.')
    ASPECT_BATCH_DIR.mkdir(exist_ok=True)
    for batch in tqdm(batches,desc='GPT-4o mini: metadata only'):
        call_no = attempts+1
        request_body = {'products':[metadata_for_llm(item_id) for item_id in batch]}
        messages = [{'role':'system','content':SYSTEM_PROMPT},
                    {'role':'user','content':json.dumps(request_body,ensure_ascii=False)}]
        # 요청을 보내기 전에 시도를 기록해, 응답 전에 중단되어도 재시작 시 해당 시도가 예산에 포함되게 한다.
        append_jsonl(LLM_ATTEMPTS, {'call':call_no,'ids':batch,'timestamp':time.time()})
        append_jsonl(LLM_INPUTS, {'call':call_no,'model':CFG['gpt_model'],'messages':messages})
        attempts+=1
        try:
            response = api.chat.completions.create(
                model=CFG['gpt_model'], messages=messages,
                response_format={'type':'json_object'},
                max_tokens=CFG['gpt_max_output_tokens'],temperature=0)
            # 원본 응답을 먼저 보관하고 JSON을 파싱해 요청한 상품 ID가 모두 돌아왔는지 검사한다.
            raw = response.choices[0].message.content or ''
            append_jsonl(LLM_OUTPUTS,{'call':call_no,'raw_output':raw,
                                      'finish_reason':response.choices[0].finish_reason})
            parsed=json.loads(raw)
            returned={str(obj['item_id']):parse_aspects(obj) for obj in parsed['items']
                      if isinstance(obj,dict) and 'item_id' in obj}
            if not all(k in returned for k in batch):
                raise ValueError('Response does not contain all requested item IDs')
            # 검사를 통과한 배치만 체크포인트에 저장해 다음 실행에서 재사용한다.
            batch_labels = {item_id: returned[item_id] for item_id in batch}
            checkpoint = ASPECT_BATCH_DIR / f'batch_{call_no:06d}.json'
            save_aspect_cache(checkpoint, batch_labels)
            completed_batches.append(checkpoint)
            aspect_cache.update(batch_labels)
        except Exception as exc:
            append_jsonl(LLM_OUTPUTS,{'call':call_no,'error_type':type(exc).__name__,'error':str(exc)[:500]})
            print('Stopped on API/parsing failure. Resume this cell to retry remaining items; the attempt is counted.')
            raise

# 모든 상품의 레이블이 모이면 최종 캐시에 병합한 뒤 배치 체크포인트를 정리한다.
assert all(k in aspect_cache for k in ITEM_IDS), 'Aspect labeling is incomplete.'
if completed_batches:
    save_aspect_cache(ASPECT_CACHE, aspect_cache)
    for checkpoint in completed_batches:
        checkpoint.unlink(missing_ok=True)
# GPT 속성은 추정값이다. TRAIN에 gold 카테고리가 있는 상품은 그 경로를 category의 정답으로 쓴다.
ASPECTS = {k:parse_aspects(aspect_cache[k]) for k in ITEM_IDS}
for k,path in TRAIN_CATEGORY.items():
    ASPECTS[k]['category']=' > '.join(path).lower()
print('Metadata aspects ready:',len(ASPECTS),'| total recorded API attempts:',attempts)
print('Sample (no review content):',next(iter(ASPECTS.items())))

### Aspect vocabularies and weak-label masks

Single-valued attributes use cross-entropy; multi-valued **Features** use binary cross-entropy. OOV, `unknown`, or genuinely missing attributes are **masked**, not forced into an incorrect class. Metadata supervision is applied across the entire catalog in a dedicated pass. Review supervision is restricted to linked **training** reviews: category is a weak contextual target, while other aspects are included only when their literal value occurs in at least one of the five reviews. This conservative mask reduces hallucinated review supervision.

In [ ]:
# 색상이나 브랜드는 문자열로 저장돼 있다. 속성별로 자주 나온 값에 번호를 붙여 분류 헤드의 출력 열과 대응시킨다.
ASPECT_CAPS = {'category':600,'brand':400,'color':120,'material':180,'size':160,'features':450}
# 한 상품의 특정 속성값을 리스트로 반환한다. 알려진 값이 없으면 빈 리스트를 반환한다.
def labels_for_item(item_id,key):
    val=ASPECTS[item_id][key]
    if key=='features': return [v for v in val if v!='unknown']
    return [] if val=='unknown' else [val]

# 카탈로그에서 두 번 이상 등장한 값을 빈도순으로 골라, 속성마다 값과 정수 인덱스의 대응표를 만든다.
VOCABS={}
for key in ASPECT_KEYS:
    freq=Counter(v for item_id in ITEM_IDS for v in labels_for_item(item_id,key))
    ordered=[v for v,n in freq.most_common() if n>=2][:ASPECT_CAPS[key]]
    VOCABS[key]={v:i for i,v in enumerate(ordered)}
save_json({'vocabularies':VOCABS,'config':CFG}, WORK_DIR/'aspect_vocabulary_and_config.json')
print('Aspect head sizes:',{k:len(v) for k,v in VOCABS.items()})

# 상품 속성을 리뷰의 정답으로 옮길 때는 근거가 필요하다. 예를 들어 색상은 그 값이 리뷰 본문에 등장해야 쓴다.
# category는 다섯 리뷰가 같은 gold 상품을 설명한다는 연결을 이용해 묶음 전체의 레이블로 쓴다.
def aspect_supervision(item_id,reviews=None):
    result={}
    review_text=' '.join(reviews).lower() if reviews is not None else None
    for key in ASPECT_KEYS:
        vals=labels_for_item(item_id,key)
        if reviews is not None and key!='category':
            vals=[v for v in vals if len(v)>=2 and v in review_text]
        result[key]=[VOCABS[key][v] for v in vals if v in VOCABS[key]]
    return result

# 속성마다 정답이 있는 행만 골라 손실을 구한다. 빈 레이블은 해당 속성을 모른다는 뜻이므로 그 행을 건너뛴다.
# 최종값은 이번 배치에서 실제로 정답을 본 속성 헤드들의 평균 손실이다.
def aspect_loss_from_pooled(model,pooled,targets,device=DEVICE):
    total=pooled.sum()*0.; used=0
    for key,head in model.aspect_heads.items():
        # features는 한 상품에 여러 값이 가능하므로 multi-hot 정답과 BCE를 사용한다.
        if key=='features':
            rows=[i for i,t in enumerate(targets) if t.get(key)]
            if not rows: continue
            target=torch.zeros((len(rows),len(VOCABS[key])),device=device)
            for j,i in enumerate(rows): target[j,targets[i][key]]=1.
            loss=F.binary_cross_entropy_with_logits(head(pooled[rows]),target)
        # 나머지 속성은 한 행에 정답 클래스 하나를 두고 cross-entropy로 학습한다.
        else:
            rows=[i for i,t in enumerate(targets) if t.get(key)]
            if not rows: continue
            target=torch.tensor([t[key][0] for i,t in enumerate(targets) if t.get(key)],device=device)
            loss=F.cross_entropy(head(pooled[rows]),target)
        total=total+loss; used+=1
    return total/max(used,1)

## 3. Gold negative sampling and ColBERT-style model

For every gold review group, use **4 random negatives + 2 same-parent/different-leaf negatives + 2 same-leaf/different-product negatives**, with fallback to random sampling if a bucket is small. Only **training-gold** category paths define reliable category buckets; GPT-inferred categories never masquerade as gold category labels.

The retriever has independent Transformer encoders and learned token projections. Its score is the requested mean over five reviews of the sum of query-token **MaxSim** values against a metadata candidate:

$$s(R,M)=\frac15\sum_{k=1}^5\sum_{t\in r_k}\max_{j\in M}\,E_R(r_k)_t^\top E_M(M)_j.$$

Special tokens and padding are masked. Training uses a per-query listwise contrastive softmax over one positive and eight negatives. This is a **lightweight ColBERT-style implementation**, not the official ColBERT checkpoint/training library.

In [ ]:
# 카테고리 경로 전체가 같으면 같은 말단 분류이고, 마지막 단계만 다르면 같은 부모 아래의 이웃 분류다.
# TRAIN 상품을 두 기준으로 미리 묶어 두면 정답과 종류가 비슷한 오답 상품을 학습 배치에 넣기 쉽다.
CAT_TO_ITEMS=defaultdict(list); PARENT_TO_ITEMS=defaultdict(list)
for iid,path in TRAIN_CATEGORY.items():
    CAT_TO_ITEMS[path].append(iid)
    PARENT_TO_ITEMS[path[:-1]].append(iid)

# 정답 상품 A와 비교할 다른 상품 ID를 뽑는다. 이 상품들의 메타데이터가 랭킹용 오답이 된다.
def sample_negatives(pos_id, n=8, mode='mixed', rnd=None):
    rnd = rnd or random
    picked=[]; excluded={pos_id}
    # 전체 카탈로그에서 아직 선택하지 않은 상품을 요청한 개수만큼 무작위로 채운다.
    def add_random(count):
        while count>0:
            item=ITEM_IDS[rnd.randrange(len(ITEM_IDS))]
            if item in excluded: continue
            picked.append(item);excluded.add(item);count-=1
    # 주어진 카테고리 후보군에서 중복 없이 뽑는다. 후보가 부족하면 있는 만큼만 추가한다.
    def add_from(pool,count):
        options=[x for x in pool if x not in excluded]
        chosen=rnd.sample(options,min(count,len(options)))
        picked.extend(chosen); excluded.update(chosen)
    if mode=='mixed' and pos_id in TRAIN_CATEGORY:
        path=TRAIN_CATEGORY[pos_id]
        # 기본값은 무작위 4개, 부모는 같고 말단이 다른 상품 2개, 말단이 같은 상품 2개다.
        add_random(n//2)
        add_from([x for x in PARENT_TO_ITEMS[path[:-1]] if TRAIN_CATEGORY[x]!=path],(n-n//2)//2)
        add_from(CAT_TO_ITEMS[path],n-n//2-(n-n//2)//2)
    else:
        add_random(n)
    # 카테고리 안에서 필요한 수를 못 채웠으면 나머지 오답은 전체 카탈로그에서 무작위로 채운다.
    if len(picked)<n: add_random(n-len(picked))
    assert len(picked)==n and len(set(picked))==n and pos_id not in picked
    return picked

# 공통 토크나이저와 특수 토큰 ID들을 준비해 리뷰와 상품에 같은 토큰 규칙을 적용한다.
TOKENIZER=AutoTokenizer.from_pretrained(CFG['backbone'],use_fast=True)
SPECIAL_IDS=set(TOKENIZER.all_special_ids)

# 텍스트 배치를 토큰 ID와 attention mask 텐서로 바꾸고, 길이 제한과 패딩을 적용해 모델의 실행 장치로 보낸다.
def tokenize(texts,max_length):
    out=TOKENIZER(texts,padding=True,truncation=True,max_length=max_length,return_tensors='pt')
    return {k:v.to(DEVICE) for k,v in out.items()}

# 상품 설명과 리뷰를 각각 인코딩한 뒤 토큰끼리 비교하는 검색기다. 같은 상품 설명을 여러 쿼리에 재사용할 수 있다.
class ColBERTAspects(nn.Module):
    # 같은 백본으로 리뷰·상품 인코더를 따로 만들고, 각자의 토큰 투영층과 공통 속성 헤드를 준비한다.
    def __init__(self):
        super().__init__()
        self.query_encoder=AutoModel.from_pretrained(CFG['backbone'])
        self.doc_encoder=AutoModel.from_pretrained(CFG['backbone'])
        h=self.query_encoder.config.hidden_size
        self.query_proj=nn.Linear(h,CFG['dimension'],bias=False)
        self.doc_proj=nn.Linear(h,CFG['dimension'],bias=False)
        self.aspect_heads=nn.ModuleDict({k:nn.Linear(h,len(v))
                                          for k,v in VOCABS.items() if len(v)>0})
        # 속성 예측에는 공통 aspect_heads를 쓴다. 아래 Identity 모듈에는 학습할 가중치가 없다.
        self.query_to_aspects=nn.Identity()
        self.doc_to_aspects=nn.Identity()
        # 리뷰 쌍 분류 전용 학습 파라미터는 내적 점수에 더하는 스칼라 편향 b 하나다.
        # 리뷰 벡터는 기존 query_encoder와 query_proj에서 오므로 별도의 임베딩 테이블이나 MLP가 필요하지 않다.
        self.pair_bias = nn.Parameter(torch.zeros(()))
        self.pair_bce = nn.BCEWithLogitsLoss()

    # 한 번 인코딩해 랭킹·리뷰 쌍 분류용 토큰 벡터와 속성 분류용 hidden 평균을 함께 얻는다.
    # 마지막 반환값은 어떤 위치가 유효한 토큰인지를 나타낸다. 평균과 MaxSim을 계산할 때 같은 위치를 선택한다.
    def encode(self,tokens,side):
        backbone = self.query_encoder if side=='query' else self.doc_encoder
        projection = self.query_proj if side=='query' else self.doc_proj
        hidden = backbone(**tokens).last_hidden_state
        # 패딩과 특수 토큰 위치를 표시해 이후 평균과 MaxSim에서 사용할 본문 토큰을 정한다.
        mask=tokens['attention_mask'].bool()
        special=torch.zeros_like(mask)
        for t in SPECIAL_IDS: special |= tokens['input_ids'].eq(t)
        valid=mask & ~special
        # 본문 토큰이 없으면 attention mask를 써서 [CLS]·[SEP]처럼 패딩이 아닌 토큰을 살린다.
        empty=~valid.any(dim=-1)
        if empty.any(): valid[empty]=mask[empty]
        # 속성 분류에는 투영 전 hidden 평균을, 랭킹과 리뷰 쌍 분류에는 투영·정규화한 토큰 벡터를 사용한다.
        pooled=(hidden*valid.unsqueeze(-1)).sum(1)/valid.sum(1,keepdim=True).clamp(min=1)
        projected=F.normalize(projection(hidden),p=2,dim=-1)
        return projected,pooled,valid

    # MaxSim은 각 리뷰 토큰이 후보 상품에서 가장 잘 맞는 토큰 하나를 골라 그 내적을 점수로 쓰는 연산이다.
    # 이 값들을 리뷰 안에서 합하고, 다섯 리뷰의 점수를 평균내 각 상품 후보의 점수를 구한다.
    def scores(self,q_emb,q_mask,d_emb,d_mask):
        # q_emb는 [배치, 리뷰 5개, 리뷰 토큰, 차원], d_emb는 [배치, 후보 상품, 상품 토큰, 차원]이다.
        dots=torch.einsum('brqd,bcmd->brcqm',q_emb,d_emb)
        dots=dots.masked_fill(~d_mask[:,None,:,None,:],-1e4)
        maxima=dots.max(-1).values
        maxima=maxima*q_mask[:,:,None,:]
        return maxima.sum(-1).mean(1)  # 출력 크기 [배치, 후보 상품 수]

# 재랭커와 baseline은 리뷰 다섯 개를 한 텍스트로 읽는다. 각 리뷰를 자른 뒤 번호와 [SEP]로 경계를 표시한다.
def joined_reviews(reviews):
    # 리뷰별 상한을 먼저 적용한다. 재랭커가 상품과 합쳐 다시 자를 때 뒤쪽 리뷰가 더 짧아질 수 있다.
    chunks=[]
    for i,r in enumerate(reviews):
        ids=TOKENIZER.encode(r,add_special_tokens=False,truncation=True,
                             max_length=CFG['cross_tokens_per_review'])
        chunks.append(f'Review {i+1}: '+TOKENIZER.decode(ids,skip_special_tokens=True))
    return ' [SEP] '.join(chunks)

# 한 행을 리뷰 5개, 목표 상품 1개, 오답 상품 n개로 만든다.
# 목표 상품을 첫 열에 놓으므로 뒤의 랭킹 손실에서 정답 인덱스는 항상 0이다.
def make_train_batch(rows,rnd):
    reviews=[r['reviews'] for r in rows]
    docs=[[r['item_id']]+sample_negatives(r['item_id'],CFG['negative_count'],CFG['negative_mode'],rnd)
          for r in rows]
    qtokens=tokenize([t for five in reviews for t in five],CFG['query_tokens'])
    dtokens=tokenize([CAT_TEXTS[ITEM_TO_IDX[iid]] for candidates in docs for iid in candidates],
                     CFG['metadata_tokens'])
    return reviews,docs,qtokens,dtokens

# 전체 카탈로그의 메타데이터로 상품 인코더와 속성 헤드를 먼저 학습한다. 리뷰·상품 연결은 필요하지 않다.
def train_metadata_aspects(model,optimizer):
    if not CFG['aspect_enabled']: return
    print('Metadata auxiliary training: every catalog item, not only labeled pairs.')
    model.train()
    for epoch in range(CFG['aspect_epochs']):
        order=np.random.default_rng(SEED+epoch).permutation(len(ITEM_IDS))
        bar=tqdm(range(0,len(order),CFG['aspect_batch']),desc=f'metadata aspect epoch {epoch+1}')
        for start in bar:
            idx=order[start:start+CFG['aspect_batch']]
            tok=tokenize([CAT_TEXTS[i] for i in idx],CFG['metadata_tokens'])
            targets=[aspect_supervision(ITEM_IDS[i]) for i in idx]
            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                _,pooled,_=model.encode(tok,'doc')
                loss=aspect_loss_from_pooled(model,pooled,targets)
            if loss.requires_grad:
                loss.backward();nn.utils.clip_grad_norm_(model.parameters(),CFG['max_grad_norm']);optimizer.step()
        print('Completed metadata aspect epoch',epoch+1)
        # 메타데이터 속성 학습 Checkpoint 저장
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, CKPT_DIR / 'metadata_aspect_last.pt')

# 리뷰 쌍 분류의 음성 예시는 다른 상품에 달린 리뷰다.
# 배치에 gold 상품이 한 종류만 들어온 경우를 위해 TRAIN의 gold 리뷰를 상품별로 모은다.
def make_gold_negative_sampler(training_rows):
    """Build a uniform sampler over the other gold products, preserving their input order."""
    groups_by_item = defaultdict(list)
    for row in training_rows:
        if not row.get('is_pseudo', False):
            groups_by_item[row['item_id']].append(row['reviews'])
    item_ids = tuple(groups_by_item)
    item_positions = {item_id: index for index, item_id in enumerate(item_ids)}

    # 현재 상품을 제외하고 상품 하나를 균등 추출한 뒤 그 상품의 리뷰 묶음과 리뷰 한 개를 차례로 뽑는다.
    def sample_other_review(item_id, rng):
        excluded = item_positions.get(item_id)
        eligible_count = len(item_ids) - int(excluded is not None)
        if eligible_count == 0:
            return None
        # 현재 상품의 위치를 건너뛰면 전체 후보 목록을 매번 다시 만들지 않고 같은 순서에서 추출할 수 있다.
        index = rng.randrange(eligible_count)
        if excluded is not None and index >= excluded:
            index += 1
        group = rng.choice(groups_by_item[item_ids[index]])
        return rng.choice(group)

    return sample_other_review


# 리뷰 두 개를 비교하기 위해 랭킹에 쓴 유효 토큰 벡터를 평균내고, 리뷰별 벡터의 길이를 1로 맞춘다.
def pool_review_tokens(token_embeddings, valid_tokens):
    """Convert [..., tokens, dim] to one unit-length vector per review."""
    mask = valid_tokens.unsqueeze(-1)
    mean = (token_embeddings.float() * mask).sum(dim=-2) / mask.sum(dim=-2).clamp_min(1)
    return F.normalize(mean, dim=-1)


# 상품 A의 리뷰 A1과 A2는 정답 1이고, 상품 A의 A1과 상품 B의 B1은 정답 0이다.
# 이 이진 분류는 리뷰 표현을 학습시키는 보조 과제다. 제출 순위는 뒤의 검색기와 재랭커가 계산한다.
def review_pair_loss(model, query_embeddings, query_mask, rows, sample_other_review, pair_rng):
    """Balance the mean BCE of same-group positives and different-product gold negatives."""
    # pseudo 행을 먼저 제외한다. gold 행이 하나도 없으면 이 배치의 pair loss 기여도는 0이다.
    gold_rows = [i for i, row in enumerate(rows) if not row.get('is_pseudo', False)]
    if not gold_rows:
        return query_embeddings[:0].sum(dtype=torch.float32)

    # query_id는 묶음을 식별한다. 두 리뷰가 서로 다른 상품인지 판정할 때는 gold의 item_id를 비교한다.
    item_ids = [rows[i]['item_id'] for i in gold_rows]
    different_products = [(i, j) for i, j in itertools.combinations(range(len(item_ids)), 2)
                          if item_ids[i] != item_ids[j]]
    other_vector = None
    # gold 상품이 한 종류면 TRAIN의 다른 상품 리뷰 하나만 더 인코딩한다. 다른 상품이 없으면 pair loss는 0이다.
    if not different_products:
        other_review = sample_other_review(item_ids[0], pair_rng)
        if other_review is None:
            return query_embeddings[:0].sum(dtype=torch.float32)
        tokens = tokenize([other_review], CFG['query_tokens'])
        extra_embeddings, _, extra_mask = model.encode(tokens, 'query')
        other_vector = pool_review_tokens(extra_embeddings, extra_mask)[0]

    if len(gold_rows) != len(rows):
        query_embeddings = query_embeddings[gold_rows]
        query_mask = query_mask[gold_rows]
    vectors = pool_review_tokens(query_embeddings, query_mask)  # 크기 [gold 묶음 수, 리뷰 5개, 임베딩 차원]
    groups, reviews_per_group, _ = vectors.shape
    flat = vectors.flatten(0, 1)

    # 혼합 정밀도 사용 중에도 리뷰 벡터 간 내적과 분류 손실은 float32로 계산한다.
    # blocks[g, h]의 5×5 원소는 g번 묶음의 각 리뷰와 h번 묶음의 각 리뷰 사이의 유사도다.
    with torch.autocast(device_type=flat.device.type, enabled=False):
        similarities = flat @ flat.T
        blocks = similarities.reshape(groups, reviews_per_group, groups, reviews_per_group).transpose(1, 2)
        group_index = torch.arange(groups, device=flat.device)
        within_group = blocks[group_index, group_index]
        left, right = torch.triu_indices(reviews_per_group, reviews_per_group, offset=1, device=flat.device)
        positive_scores = within_group[:, left, right].flatten()  # 대각선을 뺀 상삼각의 10쌍
        # 서로 다른 두 상품의 묶음에서는 5×5=25개 조합을 모두 negative로 쓴다.
        if different_products:
            first, second = zip(*different_products)
            negative_scores = blocks[list(first), list(second)].flatten()
        else:
            negative_scores = flat @ other_vector

        # 리뷰 벡터 내적을 온도로 나눈 뒤 편향 b를 더한다. BCEWithLogitsLoss가 sigmoid를 내부에서 처리한다.
        temperature = CFG['pair_temperature']
        positive_logits = positive_scores / temperature + model.pair_bias
        negative_logits = negative_scores / temperature + model.pair_bias
        positive_loss = model.pair_bce(positive_logits, torch.ones_like(positive_logits))
        negative_loss = model.pair_bce(negative_logits, torch.zeros_like(negative_logits))
        # 서로 다른 두 상품이 한 묶음씩 있으면 양성은 20쌍, 음성은 25쌍이다.
        # 두 클래스의 평균 손실에 각각 0.5를 곱해 쌍의 개수와 무관하게 비중을 맞춘다.
        return 0.5 * (positive_loss + negative_loss)


# ColBERT를 학습한다. 기본 ranking loss에 설정에 따라 상품 속성, 리뷰 속성, 리뷰 쌍 분류 손실을 가중합한다.
def fit_retriever(model, rows, epochs, checkpoint, include_review_aspects=True):
    model.to(DEVICE).train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr_retriever'], weight_decay=0.01)
    # 리뷰 쌍 손실이 켜져 있고 가중치가 0이 아닐 때만 설정을 검사하고 gold 샘플러를 준비한다.
    pair_weight = CFG['pair_loss_weight']
    use_pair_loss = CFG['pair_loss_enabled'] and pair_weight != 0
    sample_other_review = None
    if use_pair_loss:
        if not math.isfinite(CFG['pair_temperature']) or CFG['pair_temperature'] <= 0:
            raise ValueError('pair_temperature must be finite and positive.')
        if not math.isfinite(pair_weight) or pair_weight < 0:
            raise ValueError('pair_loss_weight must be finite and nonnegative.')
        sample_other_review = make_gold_negative_sampler(TRAIN)

    for epoch in range(epochs):
        # 랭킹용 난수와 리뷰 쌍용 난수를 분리해 pair loss를 켜도 랭킹에서 뽑는 오답 순서가 유지되게 한다.
        rank_rng = random.Random(SEED + epoch + 119)
        pair_rng = random.Random(SEED + epoch + 811)
        order = list(rows)
        rank_rng.shuffle(order)
        epoch_loss = torch.zeros((), dtype=torch.float64, device=DEVICE)
        batch_count = 0
        starts = range(0, len(order), CFG['retriever_batch'])
        for start in tqdm(starts, desc=f'retriever epoch {epoch + 1}'):
            batch = order[start:start + CFG['retriever_batch']]
            _, candidates, query_inputs, doc_inputs = make_train_batch(batch, rank_rng)
            batch_size = len(batch)
            candidate_count = 1 + CFG['negative_count']
            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                # 리뷰 5개와 후보 상품들을 각자의 인코더로 처리한 뒤 배치·리뷰·후보 축을 복원한다.
                query_tokens, query_hidden_mean, query_mask = model.encode(query_inputs, 'query')
                doc_tokens, doc_hidden_mean, doc_mask = model.encode(doc_inputs, 'doc')
                query_tokens = query_tokens.reshape(batch_size, 5, query_tokens.size(1), -1)
                query_mask = query_mask.reshape(batch_size, 5, -1)
                doc_tokens = doc_tokens.reshape(batch_size, candidate_count, doc_tokens.size(1), -1)
                doc_mask = doc_mask.reshape(batch_size, candidate_count, -1)

                # 후보 0번을 정답으로 삼아 MaxSim 점수에 cross-entropy를 적용한다. pseudo 행에는 설정한 가중치를 준다.
                scores = model.scores(query_tokens, query_mask, doc_tokens, doc_mask)
                positive_index = torch.zeros(batch_size, dtype=torch.long, device=DEVICE)
                rank_loss = F.cross_entropy(scores.float() / CFG['rank_temperature'], positive_index, reduction='none')
                weights = torch.tensor([row.get('loss_weight', 1.) for row in batch], device=DEVICE)
                loss = (rank_loss * weights).mean()

                # 상품 속성 손실은 후보 메타데이터의 hidden 평균과 해당 상품의 속성 레이블을 비교한다.
                if CFG['aspect_enabled']:
                    doc_targets = [aspect_supervision(item) for group in candidates for item in group]
                    doc_loss = aspect_loss_from_pooled(model, doc_hidden_mean, doc_targets)
                    loss = loss + CFG['meta_aspect_weight'] * doc_loss
                    # 다섯 리뷰의 hidden 평균을 모아 묶음의 속성을 예측한다. pseudo 행의 속성 정답은 비워 둔다.
                    if include_review_aspects:
                        review_targets = [aspect_supervision(row['item_id'], row['reviews'])
                                          if not row.get('is_pseudo') else {} for row in batch]
                        review_hidden_mean = query_hidden_mean.reshape(batch_size, 5, -1).mean(1)
                        review_loss = aspect_loss_from_pooled(model, review_hidden_mean, review_targets)
                        loss = loss + CFG['review_aspect_weight'] * review_loss
                # 앞서 랭킹에 쓴 query_tokens를 그대로 넘긴다. 아래에서 전체 손실을 역전파하면
                # pair loss의 gradient도 이 토큰을 만든 리뷰 투영층과 인코더까지 전달된다.
                if use_pair_loss:
                    pair_loss = review_pair_loss(model, query_tokens, query_mask, batch, sample_other_review, pair_rng)
                    loss = loss + pair_weight * pair_loss

            # 합산한 손실을 한 번 역전파하고 gradient norm을 제한한 뒤 모델을 갱신한다.
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG['max_grad_norm'])
            optimizer.step()
            epoch_loss += loss.detach()
            batch_count += 1
        mean_loss = epoch_loss.item() / batch_count if batch_count else math.nan
        print(f'Retriever epoch {epoch + 1}: mean loss {mean_loss:.4f}')
         # 매 Epoch 종료 시 학습 재개용 Checkpoint 저장
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, CKPT_DIR / f'{Path(checkpoint).stem}_last.pt')
    torch.save(model.state_dict(), CKPT_DIR/checkpoint)
    return model


## 4. Stage 1 — supervised ColBERT fine-tuning

First, cover **all 50k metadata examples** with auxiliary aspect supervision. Then fine-tune on gold `(reviews, positive metadata, random/category-aware negatives)` using the existing ranking and aspect objectives plus a gold-only review-pair auxiliary objective:

$$\mathcal L = \mathcal L_{ranking}+\lambda_m\mathcal L^M_{aspect}+\lambda_r\mathcal L^R_{aspect}+\lambda_p\mathcal L_{pair}.$$

**Review–review same-product classification.** Reuse the projected token embeddings and valid-token masks from the ranking forward pass. For each individual review, compute

$$z_i=\operatorname{Normalize}\left(\frac{\sum_t m_{it}E_R(r_i)_t}{\max(1,\sum_t m_{it})}\right),\qquad p_{ij}=\sigma(z_i^\top z_j/\tau+b).$$

The existing encoder mask excludes padding and special tokens, retaining its original empty-text fallback. Only the scalar bias $b$ is added; there is **no additional embedding table or MLP**. `BCEWithLogitsLoss` receives the logits directly, and gradients flow through pooling/normalization to the review token projection and Transformer encoder.

- **Positive:** all 10 unordered pairs within each gold group of five reviews.
- **Negative:** all unordered in-batch gold-review pairs with different `item_id` values. Different `query_id` values never imply different products; same-item cross-query pairs are excluded from negatives.
- **Balance:** $\mathcal L_{pair}=\tfrac12\operatorname{mean}(\mathrm{BCE}_{positive})+\tfrac12\operatorname{mean}(\mathrm{BCE}_{negative})$.
- **Small/mixed batches:** when the batch contains gold reviews from only one product, encode one additional review from a different **training-gold** product for negatives only. A separate RNG leaves the ranking-negative sampler unchanged. If no different training-gold product exists, skip the pair objective for that batch. Pseudo-only batches contribute zero pair loss.

Defaults: `pair_loss_enabled=True`, `pair_loss_weight=0.1`, `pair_temperature=0.1`. For a controlled pair-loss ablation, set `pair_loss_enabled=False` (or `pair_loss_weight=0.0`) and rerun from a fresh model with the same seed and split. Both settings skip pair computation and extra encoding entirely. The auxiliary head is unused at inference.

For the negative-sampling ablation set `negative_mode='random'`; for the multi-aspect ablation set `aspect_enabled=False`. Pair classification remains independently configurable. Record all settings alongside results.


In [ ]:
# 먼저 상품 설명만으로 속성을 맞히게 한다. 이 단계에서는 리뷰 연결이 없는 카탈로그 상품도 학습에 쓸 수 있다.
retriever=ColBERTAspects().to(DEVICE)
optimizer_aspect=torch.optim.AdamW(retriever.parameters(),lr=CFG['lr_aspect'])
train_metadata_aspects(retriever,optimizer_aspect)
del optimizer_aspect
# 다음에는 gold 리뷰로 정답 상품을 찾도록 같은 검색기를 미세조정하고 그 가중치를 저장한다.
retriever=fit_retriever(retriever,TRAIN,CFG['retriever_epochs'],'colbert_gold.pt')

## 5. Stage 2 — supervised cross-encoder reranking

The cross-encoder takes the five reviews as one sequence and the product metadata as the other. Its joint `longest_first` truncation can shorten the review sequence when both inputs are long, including all tokens from a trailing review. It learns one scalar relevance score from the retained tokens. Each mini-batch contains the same random + category-aware negative mixture. A listwise softmax pushes the gold product above the distractors. This implementation deliberately leaves auxiliary aspect supervision off the cross-encoder; assess that as a separate ablation if time permits.

$$\mathcal L_{cross}=-\log\frac{e^{s(R,M^+)}}{\sum_{M\in\{M^+\}\cup\mathcal N} e^{s(R,M)}}.$$

In [ ]:
# 검색기가 추린 후보마다 다섯 리뷰와 해당 상품 설명을 함께 읽고, 관련도 점수 하나를 내는 재랭커다.
class CrossReranker(nn.Module):
    # 사전학습 Transformer 위에 점수 하나를 출력하는 분류 헤드를 갖춘 모델을 불러온다.
    def __init__(self):
        super().__init__()
        self.net=AutoModelForSequenceClassification.from_pretrained(CFG['backbone'],num_labels=1)

    # 각 리뷰 묶음과 후보 메타데이터를 한 쌍으로 인코딩해, 쌍마다 관련도 점수(logit) 하나를 반환한다.
    def score(self,reviews,doc_texts):
        # 총길이가 cross_tokens를 넘으면 더 긴 쪽부터 줄인다. 이때 뒤쪽 리뷰 전체가 잘릴 수도 있다.
        inputs=TOKENIZER(reviews,doc_texts,padding=True,truncation='longest_first',
                         max_length=CFG['cross_tokens'],return_tensors='pt')
        inputs={k:v.to(DEVICE) for k,v in inputs.items()}
        return self.net(**inputs).logits.reshape(-1)

# 검색기와 재랭커 모두 후보 중 정답을 맞히는 cross-entropy로 랭킹을 배운다.
# 검색기의 입력 점수는 토큰 MaxSim이고, 여기서 쓰는 점수는 리뷰와 설명을 함께 읽은 cross-encoder의 logit이다.
def fit_cross(model,rows,epochs,checkpoint):
    model.to(DEVICE).train()
    optimizer=torch.optim.AdamW(model.parameters(),lr=CFG['lr_cross'],weight_decay=.01)
    n=CFG['negative_count']+1
    for epoch in range(epochs):
        rnd=random.Random(SEED+epoch+233)
        order=list(rows);rnd.shuffle(order)
        losses=[]
        for start in tqdm(range(0,len(order),CFG['cross_batch']),desc=f'cross epoch {epoch+1}'):
            batch=order[start:start+CFG['cross_batch']]
            docs=[[r['item_id']]+sample_negatives(r['item_id'],CFG['negative_count'],CFG['negative_mode'],rnd)
                  for r in batch]
            # 같은 리뷰 묶음을 후보 수만큼 반복해 각 후보 상품과 1:1 입력 쌍을 만든다.
            query_groups=[joined_reviews(r['reviews']) for r in batch]
            queries=[q for q in query_groups for _ in range(n)]
            metas=[CAT_TEXTS[ITEM_TO_IDX[iid]] for group in docs for iid in group]
            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                # logit을 [배치, 후보 수]로 모은다. 정답은 검색기와 같은 0번 열이고, CE에는 재랭커 원점수를 넣는다.
                scores=model.score(queries,metas).reshape(len(batch),n)
                targets=torch.zeros(len(batch),dtype=torch.long,device=DEVICE)
                losses_per=F.cross_entropy(scores.float(),targets,reduction='none')
                weights=torch.tensor([r.get('loss_weight',1.) for r in batch],device=DEVICE)
                loss=(losses_per*weights).mean()
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),CFG['max_grad_norm']);optimizer.step()
            losses.append(float(loss.detach().cpu()))
        print(f'Cross epoch {epoch+1}: mean loss {np.mean(losses):.4f}')
        # 매 Epoch 종료 시 학습 재개용 Checkpoint 저장
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, CKPT_DIR / f'{Path(checkpoint).stem}_last.pt')
    torch.save(model.state_dict(),CKPT_DIR/checkpoint)
    return model

# gold 리뷰와 후보 상품으로 재랭커를 먼저 학습하고 cross_gold.pt 체크포인트를 만든다.
reranker=CrossReranker().to(DEVICE)
reranker=fit_cross(reranker,TRAIN,CFG['cross_epochs'],'cross_gold.pt')

## 6. Inference stage — token-level ANN → exact late interaction → cross-encoder

**Offline:** encode all catalog metadata once and save token vectors on disk, then build a FAISS inner-product inverted-file index. The index stores a mapping **token → catalog item**. FAISS proposes a large candidate pool; exact five-review MaxSim refines it to the top `candidate_topk` items. The cross-encoder then reranks these candidates. This is the standard two-stage ColBERT indexing pattern, with an extra cross-encoder stage: [ColBERT paper](https://arxiv.org/abs/2004.12832).

**Practical distinction:** FAISS scores for individual token hits are only a **candidate-generation approximation**. The exact MaxSim formula—not token vote count—is used for the candidate shortlist. A larger ANN pool, more probes, and deeper cross-encoder top-K can improve Recall@10 at additional latency/cost.

Disk/RAM warning: 50k products × 112 tokens × 64 dimensions can produce several gigabytes of artifacts/index memory. Indexes are **recreated** after updating retriever weights; stale encoded metadata must never be reused.

In [ ]:
# FAISS가 리뷰 토큰과 가까운 상품 토큰을 찾아 후보를 좁히면, 뒤에서 각 후보를 정밀하게 채점한다.
import faiss
faiss.omp_set_num_threads(min(8,os.cpu_count() or 4))

# 여러 쿼리가 같은 상품을 검색하므로 상품은 한 번만 인코딩한다.
# 토큰 벡터를 파일에 저장하고, 검색된 각 토큰이 어느 상품에서 왔는지 찾을 수 있게 대응표를 만든다.
@torch.inference_mode()
def build_token_index(model,label,force=True):
    folder=WORK_DIR/f'index_{label}'
    folder.mkdir(parents=True,exist_ok=True)
    epath=folder/'doc_embeddings.npy';mpath=folder/'doc_masks.npy'
    ipath=folder/'token_ivf.index';tpath=folder/'token_to_item.npy'
    # 재생성을 끄고 필요한 파일이 모두 있으면 저장된 벡터·마스크·인덱스·대응표를 불러온다.
    if not force and all(p.exists() for p in (epath,mpath,ipath,tpath)):
        emb=np.load(epath,mmap_mode='r');mask=np.load(mpath,mmap_mode='r')
        return {'index':faiss.read_index(str(ipath)),'token2doc':np.load(tpath),
                'emb':emb,'mask':mask,'label':label}
    model.eval()
    # 상품 수×최대 토큰 수 크기의 파일 기반 배열을 열고, 상품을 배치별로 인코딩해 토큰 벡터와 마스크를 채운다.
    N=len(ITEM_IDS); L=CFG['metadata_tokens'];D=CFG['dimension']
    emb=np.lib.format.open_memmap(epath,mode='w+',dtype=np.float16,shape=(N,L,D))
    mask=np.lib.format.open_memmap(mpath,mode='w+',dtype=np.bool_,shape=(N,L))
    for begin in tqdm(range(0,N,CFG['index_doc_batch']),desc=f'Encode catalog: {label}'):
        end=min(N,begin+CFG['index_doc_batch'])
        tok=tokenize(CAT_TEXTS[begin:end],L)
        with amp_context(): rep,_,valid=model.encode(tok,'doc')
        v=rep.float().cpu().numpy().astype(np.float16)
        m=valid.cpu().numpy()
        emb[begin:end,:,:]=0;mask[begin:end,:]=False
        emb[begin:end,:v.shape[1],:]=v
        mask[begin:end,:m.shape[1]]=m
    emb.flush();mask.flush()

    # 전체 유효 토큰에서 최대 12만 개를 뽑아 FAISS가 검색 공간을 나눌 군집을 학습하는 데 쓴다.
    valid_docs,valid_positions=np.where(mask)
    sample_rng=np.random.default_rng(SEED)
    n_train=min(120000,len(valid_docs))
    sample=sample_rng.choice(len(valid_docs),size=n_train,replace=False)
    train_vectors=np.ascontiguousarray(emb[valid_docs[sample],valid_positions[sample]].astype('float32'))
    # 유효 토큰을 상품 행 순서로 넣으므로 token2doc[토큰 ID]로 해당 상품의 카탈로그 행을 찾을 수 있다.
    token2doc = valid_docs.astype(np.int32)
    del valid_docs, valid_positions
    # 내적 기반 IVF 인덱스의 군집 수를 정하고, 검색 시 살펴볼 군집 수를 nprobe에 설정한다.
    nlist=min(CFG['ann_nlist'],max(1,n_train//40))
    index=faiss.IndexIVFFlat(faiss.IndexFlatIP(D),D,nlist,faiss.METRIC_INNER_PRODUCT)
    index.train(train_vectors)
    index.nprobe=min(CFG['ann_nprobe'],nlist)
    # 현재 블록의 유효 토큰만 한 번에 인덱스에 추가해 전체 토큰 벡터를 한꺼번에 복사하는 비용을 줄인다.
    for begin in tqdm(range(0,N,CFG['index_add_docs']),desc='Build token ANN'):
        end=min(begin+CFG['index_add_docs'],N)
        submask=np.asarray(mask[begin:end]).reshape(-1)
        vectors=np.asarray(emb[begin:end]).reshape(-1,D)[submask]
        if len(vectors):
            index.add(np.ascontiguousarray(vectors,dtype=np.float32))
    # 인덱스 토큰 수와 대응표 길이를 확인한 뒤 인덱스, 대응표, 검색 설정을 저장한다.
    assert index.ntotal==len(token2doc)
    faiss.write_index(index,str(ipath));np.save(tpath,token2doc)
    save_json({'label':label,'n_items':N,'n_tokens':len(token2doc),
               'dimension':D,'doc_length':L,'nlist':nlist,'nprobe':index.nprobe},folder/'index_info.json')
    print('FAISS token vectors:',index.ntotal)
    return {'index':index,'token2doc':token2doc,'emb':emb,'mask':mask,'label':label}

# 리뷰 다섯 개를 각각 인코딩해 토큰 벡터와 마스크를 NumPy 배열로 반환한다. 리뷰 구분은 그대로 남긴다.
@torch.inference_mode()
def encode_five_reviews(model,revs):
    model.eval()
    assert len(revs)==5
    tok=tokenize(revs,CFG['query_tokens'])
    with amp_context(): rep,_,valid=model.encode(tok,'query')
    return rep.float().cpu().numpy(),valid.cpu().numpy()

# 기본 설정에서는 토큰 이웃으로 상품 후보를 최대 450개 모은다.
# 후보의 전체 유효 토큰으로 MaxSim을 다시 계산해 상위 50개를 재랭커에 넘긴다.
@torch.inference_mode()
def retrieve_exact(model,index_bundle,reviews,topk=None,with_review_scores=False):
    topk=topk or CFG['candidate_topk']
    q_emb,q_mask=encode_five_reviews(model,reviews)  # 벡터 크기 [리뷰 5개, 토큰 수, 임베딩 차원]
    # 유효한 리뷰 토큰 각각에 대해 가까운 상품 토큰을 찾아 유사도와 토큰 ID를 얻는다.
    q_tokens=q_emb[q_mask]
    distances,token_hits=index_bundle['index'].search(
        np.ascontiguousarray(q_tokens,dtype=np.float32),CFG['ann_per_token'])
    # 각 리뷰 토큰에서 같은 상품을 여러 번 찾으면 그 상품의 가장 높은 유사도만 남긴다.
    # 그 최댓값의 양수 부분을 상품별로 합산해 정확한 MaxSim을 계산할 후보를 고른다.
    rough=defaultdict(float)
    token2doc=index_bundle['token2doc']
    for similarities,ids in zip(distances,token_hits):
        best={}
        for sim,tid in zip(similarities,ids):
            if tid<0: continue
            doc=int(token2doc[tid]);best[doc]=max(best.get(doc,-1e6),float(sim))
        for doc,sim in best.items(): rough[doc]+=max(0.,sim)
    preliminary=[i for i,_ in sorted(rough.items(),key=lambda x:(-x[1],x[0]))[:CFG['ann_pool']]]
    # 서로 다른 상품이 topk개도 모이지 않으면 아직 뽑히지 않은 카탈로그 행으로 후보를 보충한다.
    if len(preliminary)<topk:
        seen=set(preliminary)
        preliminary.extend(i for i in range(len(ITEM_IDS)) if i not in seen)
        preliminary=preliminary[:max(topk,CFG['ann_pool'])]
    # 추린 상품들의 토큰 벡터를 읽어, 유효한 문서 토큰 전체에 대한 MaxSim을 후보별로 다시 계산한다.
    qe=torch.as_tensor(q_emb,device=DEVICE)
    qm=torch.as_tensor(q_mask,device=DEVICE)
    total=[];by_review=[]
    for offset in range(0,len(preliminary),96):
        ix=preliminary[offset:offset+96]
        de=torch.as_tensor(np.asarray(index_bundle['emb'][ix]).astype(np.float32),device=DEVICE)
        dm=torch.as_tensor(np.asarray(index_bundle['mask'][ix]),device=DEVICE)
        with amp_context():
            sims=torch.einsum('rqd,cmd->rcqm',qe,de)
            sims=sims.masked_fill(~dm[None,:,None,:],-1e4)
            review_scores=(sims.max(-1).values*qm[:,None,:]).sum(-1)  # 크기 [리뷰 5개, 현재 블록의 후보 수]
        by_review.append(review_scores.float().cpu().numpy())
        total.append(review_scores.mean(0).float().cpu().numpy())
    # 리뷰별 점수와 다섯 리뷰 평균을 모아 정렬한다. 요청하면 후속 합의도 계산에 쓸 리뷰별 점수도 돌려준다.
    allscores=np.concatenate(total)
    per_review=np.concatenate(by_review,axis=1)
    order=np.argsort(-allscores,kind='stable')[:topk]
    ids=[ITEM_IDS[preliminary[i]] for i in order]
    if with_review_scores: return ids,allscores[order],per_review[:,order]
    return ids,allscores[order]

# 후보 상품별로 cross-encoder 점수를 배치 계산하고, 점수가 높은 순서대로 상품 ID와 점수를 함께 반환한다.
@torch.inference_mode()
def rerank_candidate_ids(model,reviews,ids):
    model.eval()
    q=joined_reviews(reviews)
    result=[]
    for start in range(0,len(ids),CFG['cross_infer_batch']):
        sub=ids[start:start+CFG['cross_infer_batch']]
        with amp_context():
            logits=model.score([q]*len(sub),[CAT_TEXTS[ITEM_TO_IDX[iid]] for iid in sub])
        result.extend(logits.float().cpu().numpy().tolist())
    order=np.argsort(-np.asarray(result),kind='stable')
    return [ids[i] for i in order],[result[i] for i in order]

# 리뷰 쿼리 하나를 검색·재랭킹해 상품 10개를 반환한다. include_details를 켜면 순위와 margin도 담는다.
def predict_one(reviews,model,index_bundle,cross,include_details=False):
    candidates,colbert_scores,per_review=retrieve_exact(
        model,index_bundle,reviews,CFG['candidate_topk'],with_review_scores=True)
    ranked,ce_scores=rerank_candidate_ids(cross,reviews,candidates)
    result={'top10':ranked[:10]}
    # 검증과 pseudo 선별에서는 재랭커 1위 상품이 검색기에서도 높은 순위인지 확인한다.
    # 여기에 재랭커 1·2위의 점수 차이와 개별 리뷰들의 합의도를 함께 기록한다.
    if include_details:
        top_id=ranked[0]
        cb_rank=candidates.index(top_id)+1
        j=candidates.index(top_id)
        # 재랭커 1위 상품이 각 리뷰의 ColBERT 점수에서도 후보 중 상위 3개에 들었는지 확인한다.
        # review_agreement는 이 조건을 충족한 리뷰의 개수로, 0부터 5까지의 값이다.
        agreement=sum(int(j in np.argsort(-per_review[k],kind='stable')[:3]) for k in range(5))
        result.update(top_id=top_id,top_score=float(ce_scores[0]),
                      margin=float(ce_scores[0]-ce_scores[1]) if len(ce_scores)>1 else 0.,
                      colbert_rank=cb_rank,review_agreement=agreement,
                      candidate_ids=candidates,candidate_scores=colbert_scores.tolist(),
                      reranked_ids=ranked,reranked_scores=ce_scores)
    assert len(result['top10'])==10 and len(set(result['top10']))==10
    return result

# gold 학습이 끝난 상품 인코더로 검증과 pseudo-label 생성에 사용할 검색 인덱스를 만든다.
gold_index=build_token_index(retriever,'gold',force=True)

### Gold validation — Recall@10 and pseudo-label calibration inputs

Use only held-out gold examples. The default validation limit makes an early run manageable; for the final report set `val_eval_limit=None` and rerun evaluation. Store per-example predictions, positive labels, top-1 margin, ColBERT/Cross agreement, and five-review consistency. **The held-out gold labels never become training/pseudo labels.**

In [ ]:
# 정답을 아는 VAL 쿼리에서 검색기와 재랭커의 적중 여부를 기록하고, 쿼리별 결과와 평균 지표를 파일에 남긴다.
def validate_pipeline(model,index_bundle,cross,limit=None,output_name='validation_gold.json'):
    evaluated=VAL if limit is None else VAL[:min(len(VAL),limit)]
    records=[]
    for row in tqdm(evaluated,desc='Gold validation: ANN + Cross'):
        out=predict_one(row['reviews'],model,index_bundle,cross,include_details=True)
        # retriever_at_k는 정답이 재랭커까지 도달했는지, correct_at_10은 최종 10위 안에 들었는지를 뜻한다.
        records.append({**out,'query_id':row['query_id'],'gold_item_id':row['item_id'],
                        'correct_at_1':out['top_id']==row['item_id'],
                        'correct_at_10':row['item_id'] in out['top10'],
                        'retriever_at_10':row['item_id'] in out['candidate_ids'][:10],
                        'retriever_at_k':row['item_id'] in out['candidate_ids']})
    # 쿼리별 적중 여부를 평균내 검색 단계와 재랭킹 단계의 지표를 각각 구한다.
    stats={k:float(np.mean([r[k] for r in records])) for k in
           ('correct_at_1','correct_at_10','retriever_at_10','retriever_at_k')}
    save_json({'statistics':stats,'records':records},WORK_DIR/output_name)
    print('Validation:',len(records),'| metrics:',stats)
    return records,stats

# gold 단계의 모델을 검증해 이후 pseudo-label 임계값을 정할 쿼리별 기록을 얻는다.
VAL_RECORDS,VAL_STATS=validate_pipeline(retriever,gold_index,reranker,CFG['val_eval_limit'])

## 7. Stage 3 — validation-calibrated pseudo-labeling (optional)

The teacher is the **frozen gold-trained** retriever + cross-encoder. It proposes labels for *unlabeled test reviews only*, which the assignment explicitly permits using in training. Selection combines cross-encoder **top-1 margin**, top-1 presence near the top of the ColBERT ranking, and agreement across the five individual reviews. A held-out-gold margin threshold is chosen to reach an empirical target precision; if validation evidence is inadequate, the notebook **skips pseudo-training** instead of inventing a safe threshold.

The student is further fine-tuned on **gold + accepted pseudo pairs** with a lower pseudo weight. Pseudo review-aspect targets are masked because no gold product link is available. Review-pair classification also uses **gold reviews only**, including any fallback negative review sampled from `TRAIN`; pseudo reviews never enter its positive or negative pairs. Finally, the token ANN index must be regenerated from the updated product encoder.

**Caution:** empirical precision on 350 held-out cases has uncertainty; increase validation coverage and inspect selection quality before trusting thousands of pseudo labels. This is self-training, not proof that the pseudo pairs are correct.

In [ ]:
# margin은 재랭커의 1위 점수에서 2위 점수를 뺀 값이다. 정답을 아는 VAL에서 어느 차이부터 1위를 믿을지 정한다.
# 기준을 만족하는 값이 없으면 무한대를 반환해 pseudo 학습을 건너뛰게 한다.
def calibrate_pseudo_margin(records):
    # 재랭커 1위 상품의 ColBERT 순위와 리뷰 합의도가 기준을 충족한 검증 쿼리부터 고른다.
    candidates=[r for r in records if r['colbert_rank']<=CFG['max_colbert_rank']
                and r['review_agreement']>=CFG['min_review_agreement']]
    if len(candidates)<CFG['calibration_min_examples']:
        print('Not enough validation examples with teacher agreement; pseudo stage disabled.')
        return float('inf')
    margins=sorted({r['margin'] for r in candidates})
    acceptable=[]
    # 관측된 margin을 하나씩 임계값으로 시험해, 통과한 쿼리들의 top-1 정답률과 표본 수를 확인한다.
    for threshold in margins:
        accepted=[r for r in candidates if r['margin']>=threshold]
        if len(accepted)<CFG['calibration_min_examples']: continue
        precision=np.mean([r['correct_at_1'] for r in accepted])
        if precision>=CFG['target_pseudo_precision']:
            acceptable.append((threshold,len(accepted),float(precision)))
    if not acceptable:
        print('Target pseudo-label precision not demonstrated on validation; skip pseudo-training.')
        return float('inf')
    # 목표 정밀도와 최소 표본 수를 충족한 임계값 중 가장 낮은 값을 골라 가능한 많은 예시를 채택한다.
    threshold,count,precision=min(acceptable,key=lambda x:x[0])
    print(f'Pseudo threshold={threshold:.3f}, accepted validation={count}, empirical precision={precision:.3f}')
    return float(threshold)

# VAL에서 정한 기준을 TEST에 적용한다. 학습된 검색기와 재랭커가 예측한 1위 상품이 pseudo 정답 후보다.
PSEUDO_MARGIN=calibrate_pseudo_margin(VAL_RECORDS)
PSEUDO=[]
if CFG['run_pseudo'] and math.isfinite(PSEUDO_MARGIN):
    unlabeled=TEST if CFG['pseudo_limit'] is None else TEST[:CFG['pseudo_limit']]
    candidates=[]
    for row in tqdm(unlabeled,desc='Teacher pseudo-labeling (unlabeled test reviews)'):
        out=predict_one(row['reviews'],retriever,gold_index,reranker,include_details=True)
        # margin·ColBERT 순위·리뷰 합의도를 모두 통과한 1위 상품을 채택한다. 검증용 상품은 제외한다.
        if (out['margin']>=PSEUDO_MARGIN and out['colbert_rank']<=CFG['max_colbert_rank']
            and out['review_agreement']>=CFG['min_review_agreement']
            and out['top_id'] not in val_items):
            candidates.append({'query_id':row['query_id'],'reviews':row['reviews'],
                               'item_id':out['top_id'],'is_pseudo':True,
                               'loss_weight':CFG['pseudo_loss_weight'],'margin':out['margin']})
    # 통과한 예시를 margin 순으로 정렬해 설정한 최대 개수만 보관하고 선택 결과를 기록한다.
    candidates.sort(key=lambda r:-r['margin'])
    PSEUDO=candidates[:CFG['pseudo_max_selected']]
    save_json({'threshold':PSEUDO_MARGIN,'count':len(PSEUDO),
               'pairs':[{'query_id':x['query_id'],'item_id':x['item_id'],
                         'margin':x['margin']} for x in PSEUDO]},
              WORK_DIR/'pseudo_pairs.json')
print('Accepted pseudo pairs:',len(PSEUDO))

if PSEUDO:
    MIXED=TRAIN+PSEUDO
    # pseudo의 item_id는 모델 예측이므로 랭킹 손실에 작은 가중치를 준다.
    # 리뷰 쌍 분류는 이 혼합 배치에서도 정답 item_id가 확인된 gold 행만 사용한다.
    retriever=fit_retriever(retriever,MIXED,CFG['pseudo_epochs'],'colbert_final.pt',include_review_aspects=True)
    reranker=fit_cross(reranker,MIXED,CFG['pseudo_epochs'],'cross_final.pt')
    # 추가 학습으로 상품 인코더도 바뀌므로 최종 모델의 상품 토큰과 검색 인덱스를 다시 만든다.
    final_index=build_token_index(retriever,'pseudo_final',force=True)
else:
    print('Final student = gold-trained models (pseudo-stage not selected).')
    # 채택된 pseudo 예시가 없으면 gold 학습 단계의 검색 인덱스를 그대로 사용한다.
    final_index=gold_index

# 선택된 최종 모델과 인덱스를 같은 검증 절차로 평가해 결과를 별도 파일에 남긴다.
FINAL_VAL_RECORDS,FINAL_STATS=validate_pipeline(
    retriever,final_index,reranker,CFG['val_eval_limit'],output_name='validation_final.json')

## 8. Required experiment baselines and ablations

The assignment explicitly requests **BM25**, **dense retrieval**, and **fine-tuning with learning-to-rank** in the results comparison. The first two baselines are evaluated on the same held-out product-disjoint validation split; the ColBERT/cross model above is the learning-to-rank method. A standalone **gold-fine-tuned dense dual encoder** is also provided for a fairer representation-only baseline.

For the paper, rerun the notebook with these controlled configurations (fresh model initialization each time):

| Run | Negative sampling | Aspect loss | Pair loss | Pseudo-labeling |
|---|---|---|---|---|
| ColBERT + cross | Random only | Off | Off | Off |
| + category hard negatives | Mixed | Off | Off | Off |
| + aspect learning | Mixed | On | Off | Off |
| + review-pair classification | Mixed | On | On | Off |
| + pseudo learning (pair off) | Mixed | On | Off | On |
| + pseudo learning (pair on) | Mixed | On | On | On |

Use `pair_loss_enabled` to switch the auxiliary loss independently. Compare pair off/on with all other settings fixed, including `pair_loss_weight=0.1`, `pair_temperature=0.1`, seed, split, and training budget. Pseudo labels are never used as review-pair labels.

Hierarchical category heads and a taxonomy GNN remain **optional unimplemented extensions**, not claimed ablations. If you add them, record implementation and rerun results. **Do not report placeholder results**: the code below prints only measured values.

In [ ]:
# BM25는 리뷰와 상품 설명에 겹치는 단어로 점수를 낸다. 두 입력을 같은 규칙으로 소문자 단어·숫자로 나눈다.
def tokenize_lexical(s):
    return re.findall(r'[a-z0-9]+',s.lower())

# 텍스트별로 패딩을 제외한 hidden state를 평균내고 정규화해, 텍스트 하나당 dense 벡터 하나를 반환한다.
@torch.inference_mode()
def mean_dense_encode(backbone,texts,max_len=256,batch_size=64,projection=None):
    backbone.eval()
    vectors=[]
    for start in tqdm(range(0,len(texts),batch_size),desc='Encode dense texts',leave=False):
        tok=tokenize(texts[start:start+batch_size],max_len)
        with amp_context():
            output=backbone(**tok).last_hidden_state
            mask=tok['attention_mask'].unsqueeze(-1)
            pool=(output*mask).sum(1)/mask.sum(1).clamp(min=1)
            if projection is not None: pool=projection(pool)
            pool=F.normalize(pool.float(),dim=-1)
        vectors.append(pool.cpu().numpy())
    return np.concatenate(vectors).astype(np.float32)

# 전달받은 검색 함수로 각 쿼리의 상위 10개 상품을 얻고, 그 안에 정답이 있는 쿼리의 비율(Recall@10)을 구한다.
def evaluate_baseline_ranker(predict_fn,rows,desc):
    outcomes=[]
    for row in tqdm(rows,desc=desc):
        predictions=predict_fn(row['reviews'])[:10]
        assert len(predictions)==len(set(predictions))==10
        outcomes.append(row['item_id'] in predictions)
    return float(np.mean(outcomes))

# 모델마다 다른 쿼리를 평가하면 난이도도 달라진다. 같은 검증 쿼리에서 Recall@10을 비교하도록 ID를 맞춘다.
BASELINE_RESULTS={}
evaluated_ids = ({r['query_id'] for r in VAL_RECORDS}
                 & {r['query_id'] for r in FINAL_VAL_RECORDS})
BASELINE_ROWS = [row for row in VAL if row['query_id'] in evaluated_ids]
if CFG['run_baselines']:
    BASELINE_ROWS = BASELINE_ROWS[:CFG['baseline_eval_limit']]
# 비교할 쿼리가 있으면 전체 상품 메타데이터로 BM25 색인을 만든 뒤 공통 검증 표본에서 평가한다.
if CFG['run_baselines'] and BASELINE_ROWS:
    from rank_bm25 import BM25Okapi
    print('Building BM25 vocabulary on provided catalog only...')
    bm25=BM25Okapi([tokenize_lexical(s) for s in tqdm(CAT_TEXTS,desc='BM25 indexing')])
    # 리뷰 다섯 개의 단어로 상품 메타데이터의 BM25 점수를 계산해 상위 10개 상품 ID를 반환한다.
    def bm25_predict(reviews):
        query=tokenize_lexical(joined_reviews(reviews))
        scores=bm25.get_scores(query)
        return [ITEM_IDS[i] for i in np.argsort(-scores,kind='stable')[:10]]
    BASELINE_RESULTS['BM25']=evaluate_baseline_ranker(bm25_predict,BASELINE_ROWS,'BM25 validation')
    del bm25

    # 사전학습 백본을 그대로 사용해, 이 과제의 gold 데이터로 미세조정하기 전 dense 검색 성능을 측정한다.
    dense_base=AutoModel.from_pretrained(CFG['backbone']).to(DEVICE).eval()
    dense_docs=mean_dense_encode(dense_base,CAT_TEXTS,max_len=CFG['metadata_tokens'])
    # 합친 리뷰의 벡터와 미리 계산한 상품 벡터들의 내적을 비교해 상위 10개 상품 ID를 반환한다.
    def pretrained_dense_predict(reviews):
        q=mean_dense_encode(dense_base,[joined_reviews(reviews)],max_len=CFG['cross_tokens'],batch_size=1)
        scores=(dense_docs@q[0])
        return [ITEM_IDS[i] for i in np.argsort(-scores,kind='stable')[:10]]
    BASELINE_RESULTS['Pretrained dense']=evaluate_baseline_ranker(
        pretrained_dense_predict,BASELINE_ROWS,'Pretrained dense validation')
    del dense_base,dense_docs
    if DEVICE.type=='cuda': torch.cuda.empty_cache()

print('BM25/pretrained dense baselines:',BASELINE_RESULTS)

### Optional gold-fine-tuned dense baseline

This independent dual encoder is explicitly optimized with **listwise learning-to-rank** over the same gold positive/negative groups. Its score is cosine similarity between pooled query/metadata vectors, not token-level MaxSim. It provides a direct apples-to-apples dense-versus-ColBERT comparison. It uses a general pretrained checkpoint and gold data only.

In [ ]:
# 이 비교 모델도 gold로 학습한다. 리뷰 묶음과 상품을 각각 벡터 하나로 압축해 검색하므로
# 토큰 단위로 유사도를 계산하는 ColBERT와 같은 검증 표본에서 결과를 대조할 수 있다.
class DenseDualEncoder(nn.Module):
    # 리뷰용·상품용 인코더를 따로 만들고 각각의 출력을 128차원으로 바꾸는 투영층을 준비한다.
    def __init__(self):
        super().__init__()
        self.query_encoder=AutoModel.from_pretrained(CFG['backbone'])
        self.doc_encoder=AutoModel.from_pretrained(CFG['backbone'])
        h=self.query_encoder.config.hidden_size
        self.qproj=nn.Linear(h,128,bias=False)
        self.dproj=nn.Linear(h,128,bias=False)
    # 입력 종류에 맞는 인코더로 패딩을 제외한 평균을 구하고, 투영·정규화해 텍스트별 비교 벡터를 만든다.
    def embed(self,tokens,side):
        net,head=(self.query_encoder,self.qproj) if side=='query' else (self.doc_encoder,self.dproj)
        output=net(**tokens).last_hidden_state
        mask=tokens['attention_mask'].unsqueeze(-1)
        pool=(output*mask).sum(1)/mask.sum(1).clamp(min=1)
        return F.normalize(head(pool),dim=-1)

# gold 쿼리와 후보 상품의 내적 점수로 두 dense 인코더를 학습한 뒤 체크포인트를 저장한다.
def fit_dense(model,rows,epochs=1,batch=8):
    optimizer=torch.optim.AdamW(model.parameters(),lr=CFG['lr_retriever'])
    model.train();rnd=random.Random(SEED+777)
    n=1+CFG['negative_count']
    for ep in range(epochs):
        order=list(rows);rnd.shuffle(order)
        for start in tqdm(range(0,len(order),batch),desc=f'Fine-tune dense epoch {ep+1}'):
            subset=order[start:start+batch]
            # 각 gold 쿼리에 정답 상품 하나와 랭킹용 오답 상품들을 붙여 dense 검색기의 비교 후보를 구성한다.
            q=[joined_reviews(x['reviews']) for x in subset]
            groups=[[x['item_id']]+sample_negatives(x['item_id'],CFG['negative_count'],CFG['negative_mode'],rnd)
                    for x in subset]
            d=[CAT_TEXTS[ITEM_TO_IDX[i]] for group in groups for i in group]
            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                # 쿼리·상품 벡터의 내적을 온도 0.05로 나눈 후, 정답인 0번 후보에 cross-entropy를 적용한다.
                qe=model.embed(tokenize(q,CFG['cross_tokens']),'query')
                de=model.embed(tokenize(d,CFG['metadata_tokens']),'doc').reshape(len(subset),n,-1)
                logits=torch.einsum('bd,bnd->bn',qe,de)/0.05
                loss=F.cross_entropy(logits.float(),torch.zeros(len(subset),dtype=torch.long,device=DEVICE))
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.0);optimizer.step()
    model.eval()
    torch.save(model.state_dict(),CKPT_DIR/'dense_gold.pt')
    return model

# baseline 실행이 켜져 있고 비교 표본이 있으면 dense dual encoder를 gold 데이터로 한 에폭 학습한다.
if CFG['run_baselines'] and BASELINE_ROWS:
    dense_gold=fit_dense(DenseDualEncoder().to(DEVICE),TRAIN,epochs=1)
    # 상품마다 벡터 하나를 미리 계산해 두고, 쿼리가 들어오면 CPU에서 모든 상품과의 내적을 계산한다.
    dense_gold_docs=[]
    for start in tqdm(range(0,len(CAT_TEXTS),96),desc='Gold dense catalog index'):
        with torch.inference_mode(),amp_context():
            em=dense_gold.embed(tokenize(CAT_TEXTS[start:start+96],CFG['metadata_tokens']),'doc')
        dense_gold_docs.append(em.float().cpu().numpy())
    dense_gold_docs=np.concatenate(dense_gold_docs)
    # 학습된 dense 인코더로 리뷰를 벡터화하고, 전체 상품 벡터와의 내적이 가장 큰 상품 10개를 반환한다.
    @torch.inference_mode()
    def dense_gold_predict(reviews):
        with amp_context(): em=dense_gold.embed(tokenize([joined_reviews(reviews)],CFG['cross_tokens']),'query')
        scores=dense_gold_docs@em.float().cpu().numpy()[0]
        return [ITEM_IDS[i] for i in np.argsort(-scores,kind='stable')[:10]]
    BASELINE_RESULTS['Gold fine-tuned dense']=evaluate_baseline_ranker(
        dense_gold_predict,BASELINE_ROWS,'Fine-tuned dense validation')
    del dense_gold,dense_gold_docs
    if DEVICE.type=='cuda': torch.cuda.empty_cache()

# BM25·dense 모델과 같은 쿼리 ID들만 골라 gold 및 최종 ColBERT 파이프라인의 Recall@10을 집계한다.
for label, records in [('ColBERT + cross (gold)', VAL_RECORDS),
                       ('ColBERT + cross (final)', FINAL_VAL_RECORDS)]:
    hit_by_query = {record['query_id']: record['correct_at_10'] for record in records}
    hits = [hit_by_query[row['query_id']] for row in BASELINE_ROWS]
    BASELINE_RESULTS[label] = float(np.mean(hits)) if hits else math.nan
# 모델별 Recall@10과 공통 쿼리 수를 표로 표시하고 validation_results.csv에 저장한다.
RESULTS_TABLE = pd.DataFrame([{'model': name, 'recall_at_10': score, 'n_queries': len(BASELINE_ROWS)}
                              for name, score in BASELINE_RESULTS.items()])
display(RESULTS_TABLE)
RESULTS_TABLE.to_csv(WORK_DIR/'validation_results.csv', index=False)


## 9. Final test inference and CSV submission

For each official test query: five review encodings → token ANN hits → exact ColBERT MaxSim shortlist → cross-encoder ordering → **10 distinct IDs**. The pair-classification objective is used only during training; inference is unchanged.

The CSV format is fixed to the supplied `dummy_baseline.ipynb`: **`query_id,item_id_1,...,item_id_10`**, one row per test query in its **original input order**, with the highest-ranked item first. Submit no index column, scores, or category predictions. Set `CFG['student_id']` to your student ID; the output is `<student_id>_final.csv`. The writer reads the CSV back and checks the headers, row order, uniqueness, and catalog membership before reporting success.


In [ ]:
# dummy_baseline과 같은 열을 준비한다. 첫 열은 테스트의 query_id이고,
# 이어지는 10개 열에는 예측한 카탈로그 item_id를 1위부터 적는다.
import csv
from tempfile import TemporaryDirectory

TOP_K = 10
SUBMISSION_COLUMNS = ['query_id'] + [f'item_id_{rank}' for rank in range(1, TOP_K + 1)]


In [ ]:
# test_queries.json에서 읽은 순서로 검색한다. 한 쿼리와 그 상품 ID 10개가 제출 CSV의 한 행이다.
output_path = WORK_DIR / f"{CFG['student_id']}_final.csv"
# 결과 폴더 안의 임시 파일에 먼저 써서 도중에 실패해도 이전 제출 파일을 보존한다.
with TemporaryDirectory(dir=WORK_DIR) as staging_dir:
    temporary_csv = Path(staging_dir) / output_path.name
    with temporary_csv.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.writer(stream)
        writer.writerow(SUBMISSION_COLUMNS)
        # 쿼리 하나를 검색·재랭킹한 직후 한 행을 써서 전체 예측을 메모리에 쌓지 않는다.
        for query in tqdm(TEST, desc='Final top-10 inference'):
            ranked_ids = predict_one(query['reviews'], retriever, final_index, reranker)['top10']
            writer.writerow([query['query_id'], *ranked_ids])

    # 작성한 파일을 다시 읽어 헤더, 쿼리 순서, 상품 개수·중복·카탈로그 등록 여부를 확인한다.
    with temporary_csv.open('r', newline='', encoding='utf-8') as stream:
        reader = csv.reader(stream)
        if next(reader, None) != SUBMISSION_COLUMNS:
            raise ValueError('Unexpected submission header.')
        # strict=True로 행 수까지 대조한다. CSV의 ID는 문자열로 읽으므로 앞자리 0도 보존된다.
        for query, values in zip(TEST, reader, strict=True):
            if len(values) != len(SUBMISSION_COLUMNS) or values[0] != query['query_id']:
                raise ValueError(f"Unexpected columns or row order for query {query['query_id']}")
            ranked_ids = values[1:]
            if len(set(ranked_ids)) != TOP_K or any(i not in CATALOG for i in ranked_ids):
                raise ValueError(f"Invalid catalog IDs in the CSV for query {query['query_id']}")
    # 모든 행의 검사가 끝나면 임시 CSV를 최종 제출 경로로 옮긴다.
    temporary_csv.replace(output_path)

print(f'Saved {len(TEST):,} queries to {output_path.resolve()}')
display(pd.read_csv(output_path, nrows=3, dtype=str, keep_default_na=False))

# 설정, 검증 점수, baseline 점수, pseudo 예시 수, 제출 파일 경로를 실행 기록에 남긴다.
save_json({'config':CFG,'device':str(DEVICE),'gold_validation':VAL_STATS,
           'final_validation':FINAL_STATS,'baselines':BASELINE_RESULTS,
           'pseudo_pairs':len(PSEUDO),'output_csv':str(output_path)},WORK_DIR/'run_manifest.json')

## 10. Reproducibility and report checklist

1. Keep the exact `split.json`, `aspect_vocabulary_and_config.json`, model checkpoints, index settings, and `run_manifest.json`. Record GPU, training durations, validation sizes, and API attempts. Do not upload your API key.
2. Submit the separate `llm_inputs.jsonl` and `llm_outputs.jsonl` with the code, because the project requires **all prompts and generated outputs**. Also retain `llm_attempts.jsonl` to audit the total number of external calls.
3. The CSV writer matches `dummy_baseline.ipynb` and validates `query_id,item_id_1,...,item_id_10` in original test order. Change `student_id` from `YOUR_STUDENT_ID`, set the required Kaggle team name, then submit the CSV on Kaggle.
4. For the 8-page English report, compare BM25, dense retrieval, gold fine-tuned learning-to-rank, category-aware negatives, multi-aspect loss, review-pair classification off/on, and pseudo-labeling with **measured** Recall@10. Include success/failure cases, training strategy, and a framework figure.
5. For the final GitHub reproducibility check, use a clean environment and **run all cells in order**. Add a README with dataset placement, dependency versions, commands/environment, disk/GPU requirements, GPT expenses/remaining call budget, and model checkpoint download locations. The notebook alone does not replace that required README.

### Possible follow-up improvements (not implemented)

- Taxonomy-aware hierarchical category heads: supervise levels of `category_path` for train-gold products; mask unknown levels, avoid validation label leakage.
- Taxonomy GNN: propagate embeddings over taxonomy parent/child edges; compare against the hierarchical-head ablation first.
- Hybrid BM25 + ColBERT candidate union if pure token ANN has low gold `retriever_at_k` recall.
- Train/evaluate on additional negative ratios and `candidate_topk` values: *Recall@10 is the actual objective*, and reranking quality cannot recover a true product excluded from the candidate pool.